# Question 3


- Part 1: Corpus and Language Model Building
- Part 2: Two candidate generation approaches: Standard Edit Distance and Symmetric Delete
- Part 3: Logic for non-word and real-word error correction
- Part 4: Evaluation + the “Speed Demon” benchmark (1000 words)
- Part 5: An interactive terminal style spelling corrector CLI


## Setup — install & import

In [27]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import os

MODEL_DIR = "/content/drive/MyDrive/GROUP_ASSIGNMENT_1/models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory:", MODEL_DIR)

Model directory: /content/drive/MyDrive/GROUP_ASSIGNMENT_1/models


In [29]:
import nltk
nltk.download('brown')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import brown
from collections import Counter, defaultdict
import random, time, string

random.seed(42)
ALPHABET = string.ascii_lowercase
print("Setup complete.")

Setup complete.


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Part 1 — Corpus and Model Preparation
We construct
1. A **vocabulary** of unique lowercase alphabetic words from the Brown corpus.
2. A **unigram frequency distribution** for ranking non-word candidates.
3. **A bigram probability model with add-k smoothing**.

In [30]:
sents = brown.sents()

clean_sents = [[w.lower() for w in s if w.isalpha()] for s in sents]
clean_sents = [s for s in clean_sents if len(s) >= 2]

words = [w for s in clean_sents for w in s]
vocab = set(words)
unigram_freq = Counter(words)
total_words = sum(unigram_freq.values())

def unigram_prob(word):
    return unigram_freq.get(word, 0) / total_words

bigram_freq = Counter()
unigram_ctx_freq = Counter()
for s in clean_sents:
    tokens = ['<s>'] + s + ['</s>']
    for i in range(len(tokens) - 1):
        bigram_freq[(tokens[i], tokens[i + 1])] += 1
        unigram_ctx_freq[tokens[i]] += 1

K = 0.5
V = len(vocab) + 2
def bigram_prob(w1, w2, k=K):
    return (bigram_freq.get((w1, w2), 0) + k) / (unigram_ctx_freq.get(w1, 0) + k * V)

print(f"Vocabulary size: {len(vocab)}")
print(f"Total tokens: {total_words}")
print(f"Unique bigrams: {len(bigram_freq)}")

Vocabulary size: 40189
Total tokens: 980770
Unique bigrams: 408064


## Part 2 — Candidate Generation Methods

### Method A - Normal Edit Distance 1
All deletions, transpositions, replacements and insertions of a word filtered to those in the vocabulary.

In [31]:
def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in ALPHABET]
    inserts = [L + c + R for L, R in splits for c in ALPHABET]
    return set(deletes + transposes + replaces + inserts)

def method_a_candidates(word):
    return {c for c in edits1(word) if c in vocab}

print(method_a_candidates('teh'))

{'the', 'tex', 'tsh', 'tee', 'tea', 'tel', 'tech', 'ted', 'eh', 'ten'}


### Method B — Symmetric Delete Spelling Correction (SymSpell)

**Preprocessing (once):** for every vocabulary word,we generate all its one-character deletions and map each deletion back to the original word.

**Lookup (per query):** we generate all one-character deletions of the misspelled word and look each one up in the precomputed dictionary.

In [32]:
def build_symspell_index(vocabulary):
    index = defaultdict(set)
    for w in vocabulary:
        index[w].add(w)
        for i in range(len(w)):
            deletion = w[:i] + w[i + 1:]
            index[deletion].add(w)
    return index

print("Building SymSpell delete-index (one-time preprocessing)...")
t0 = time.time()
symspell_index = build_symspell_index(vocab)
print(f"Index built in {time.time() - t0:.2f}s — {len(symspell_index)} keys")

def method_b_candidates(word):
    deletions = {word}
    for i in range(len(word)):
        deletions.add(word[:i] + word[i + 1:])
    candidates = set()
    for d in deletions:
        if d in symspell_index:
            candidates |= symspell_index[d]
    return candidates


print(method_b_candidates('teh'))

Building SymSpell delete-index (one-time preprocessing)...
Index built in 1.02s — 310276 keys
{'the', 'tex', 'tsh', 'tie', 'tee', 'tea', 'ate', 'tel', 'tech', 'ted', 'eh', 'eph', 'ten', 'thy', 'thc', 'toe'}


## Part 3 — Spelling Correction Logic

**Non-word errors:** word is out of vocabulary -  we generate candidates - pick the one with the highest unigram frequency.

**Real-word errors:** word is in vocabulary but may be wrong in context - we compare the bigram probability of the original local phrase against edit distance 1 alternatives; switch only if a candidate scores meaningfully higher.

In [33]:
def correct_nonword(word, method='B'):
    if word in vocab:
        return word
    candidates = method_a_candidates(word) if method == 'A' else method_b_candidates(word)
    if not candidates:
        return word
    return max(candidates, key=lambda c: unigram_freq.get(c, 0))


def correct_realword(tokens, idx, method='B', threshold=2.0):
    """tokens: list of lowercase words in the sentence. idx: index of the word to check."""
    word = tokens[idx]
    if word not in vocab:
        return word, False

    prev_w = tokens[idx - 1] if idx > 0 else '<s>'
    next_w = tokens[idx + 1] if idx < len(tokens) - 1 else '</s>'
    orig_score = bigram_prob(prev_w, word) * bigram_prob(word, next_w)

    candidates = method_a_candidates(word) if method == 'A' else method_b_candidates(word)
    candidates.discard(word)

    best_word, best_score = word, orig_score
    for c in candidates:
        score = bigram_prob(prev_w, c) * bigram_prob(c, next_w)
        if score > best_score:
            best_score, best_word = score, c

    if best_word != word and best_score > orig_score * threshold:
        return best_word, True
    return word, False

print(correct_nonword('sentnce'))
print(correct_realword(['i', 'would', 'like', 'to', 'sea', 'the', 'world'], 4))

sentence
('see', True)


## Part 4 — Evaluation and "Speed Demon" Benchmark


 We take 10% of Brown corpus sentences. For each, introduce a single edit error into one randomly chosen word. Classify each generated example as a **non-word** test case  or a **real-word** test case.

In [34]:
def introduce_error(word):
    if len(word) < 2:
        return word
    ops = ['delete', 'replace', 'insert']
    if len(word) > 1:
        ops.append('transpose')
    op = random.choice(ops)
    if op == 'delete':
        i = random.randint(0, len(word) - 1)
        return word[:i] + word[i + 1:]
    elif op == 'transpose':
        i = random.randint(0, len(word) - 2)
        return word[:i] + word[i + 1] + word[i] + word[i + 2:]
    elif op == 'replace':
        i = random.randint(0, len(word) - 1)
        c = random.choice(ALPHABET)
        return word[:i] + c + word[i + 1:]
    else:
        i = random.randint(0, len(word))
        c = random.choice(ALPHABET)
        return word[:i] + c + word[i:]


eligible_sents = [s for s in clean_sents if len(s) >= 3]
random.shuffle(eligible_sents)
test_sents = eligible_sents[: max(1, int(0.1 * len(eligible_sents)))]

nonword_cases = []
realword_cases = []

for s in test_sents:
    idx = random.randint(0, len(s) - 1)
    orig = s[idx]
    corrupted = introduce_error(orig)
    tokens = s.copy()
    tokens[idx] = corrupted
    if corrupted in vocab and corrupted != orig:
        realword_cases.append((tokens, idx, orig))
    else:
        nonword_cases.append((tokens, idx, orig))

print(f"Sampled sentences: {len(test_sents)}")
print(f"Non-word test cases: {len(nonword_cases)}")
print(f"Real-word test cases: {len(realword_cases)}")

Sampled sentences: 5438
Non-word test cases: 4691
Real-word test cases: 747


### Accuracy — non-word and real-word correction

In [35]:
def evaluate_nonword(method='B'):
    correct = 0

    for tokens, idx, orig in nonword_cases:
        corrupted = tokens[idx]

        pred = correct_nonword(
            corrupted,
            method=method
        )

        if pred == orig:
            correct += 1

    return (
        correct / len(nonword_cases)
        if nonword_cases
        else 0.0
    )


def evaluate_realword(method='B'):
    correct = 0

    for tokens, idx, orig in realword_cases:
        corrupted = tokens[idx]

        pred, changed = correct_realword(
            tokens,
            idx,
            method=method
        )

        # A real-word test case is correct only when
        # the corrupted word is restored to the original.
        if pred == orig:
            correct += 1

    return (
        correct / len(realword_cases)
        if realword_cases
        else 0.0
    )


for method in ['A', 'B']:
    nonword_acc = evaluate_nonword(method)
    realword_acc = evaluate_realword(method)

    print(
        f"Method {method} — "
        f"Non-word accuracy: {nonword_acc:.2%} "
        f"(n={len(nonword_cases)})"
    )

    print(
        f"Method {method} — "
        f"Real-word accuracy: {realword_acc:.2%} "
        f"(n={len(realword_cases)})"
    )

Method A — Non-word accuracy: 84.01% (n=4691)
Method A — Real-word accuracy: 82.60% (n=747)
Method B — Non-word accuracy: 76.55% (n=4691)
Method B — Real-word accuracy: 79.25% (n=747)


### Speed Demon Benchmark

We Build 1,000 misspelled words, then run each candidate-generation method over the full batch and time it.

In [36]:
vocab_list = list(vocab)
misspelled_batch = []
while len(misspelled_batch) < 1000:
    w = random.choice(vocab_list)
    corrupted = introduce_error(w)
    if corrupted not in vocab:
        misspelled_batch.append(corrupted)

assert len(misspelled_batch) == 1000
print("Verified benchmark size:", len(misspelled_batch))

t0 = time.time()
for w in misspelled_batch:
    method_a_candidates(w)
time_a = time.time() - t0

t0 = time.time()
for w in misspelled_batch:
    method_b_candidates(w)
time_b = time.time() - t0

print(f"Method A (Edit Distance 1): {time_a:.4f}s total, {time_a/len(misspelled_batch)*1000:.4f} ms/word")
print(f"Method B (Symmetric Delete): {time_b:.4f}s total, {time_b/len(misspelled_batch)*1000:.4f} ms/word")
print(f"Speedup (A/B): {time_a / time_b:.2f}x")

Verified benchmark size: 1000
Method A (Edit Distance 1): 0.1697s total, 0.1697 ms/word
Method B (Symmetric Delete): 0.0077s total, 0.0077 ms/word
Speedup (A/B): 21.96x


**Method B** is better because in
Method A  we generate on the order of `54 * len(word) + 25` strings per query and then check each one against the vocabulary set. Method B does the expensive alphabet sized fan out only **once**, at preprocessing time, over the whole vocabulary; at query time it only needs `len(word) + 1` deletions of the misspelled word and a dictionary lookup for each. Lookup work shrinks from O(alphabet size × word length) to O(word length)

## Part 5 — Live Interactive Application

A terminal-style CLI:  we type a sentence, press Enter, see the corrected sentence with changed words wrapped in `**asterisks**`, plus the correction latency. Type `exit` to quit.

In [37]:
def correct_sentence(sentence, method='B'):
    raw_tokens = sentence.strip().split()
    lower_tokens = [''.join(ch for ch in t.lower() if ch.isalpha()) for t in raw_tokens]

    corrected = list(raw_tokens)
    changed_flags = [False] * len(raw_tokens)

    for i, clean_tok in enumerate(lower_tokens):
        if not clean_tok:
            continue
        if clean_tok not in vocab:
            fix = correct_nonword(clean_tok, method=method)
            if fix != clean_tok:
                corrected[i] = fix
                changed_flags[i] = True
        else:
            fix, changed = correct_realword(lower_tokens, i, method=method)
            if changed:
                corrected[i] = fix
                changed_flags[i] = True

    return corrected, changed_flags


def highlight(corrected, changed_flags):
    return ' '.join(f"**{w}**" if c else w for w, c in zip(corrected, changed_flags))


def run_cli(method='B'):
    print("Spelling Corrector CLI — type a sentence, or 'exit' to quit.")
    while True:
        text = input(">> ")
        if text.strip().lower() == 'exit':
            print("Goodbye!")
            break
        t0 = time.time()
        corrected, flags = correct_sentence(text, method=method)
        latency_ms = (time.time() - t0) * 1000
        print("Corrected:", highlight(corrected, flags))
        print(f"Latency: {latency_ms:.2f} ms")



for demo in [
    "I hav a good feeling about this.",
    "This is a test sentnce.",
    "I would like to sea the world.",
    "Please meat me at the station.",
]:
    corrected, flags = correct_sentence(demo)
    print(f"{demo!r} -> {highlight(corrected, flags)}")

'I hav a good feeling about this.' -> I **had** a good feeling about this.
'This is a test sentnce.' -> This is a test **sentence**
'I would like to sea the world.' -> **it** would like to **see** the world.
'Please meat me at the station.' -> Please **beat** me **to** the station.


In [38]:
%%writefile q3_cli.py

import os
import time
import cloudpickle


# ---------------------------------------------------------
# Load Q3 trained components
# ---------------------------------------------------------

MODEL_DIR = os.path.join(
    os.path.expanduser("~"),
    "GROUP_ASSIGNMENT_1",
    "models"
)

# For Colab/Drive use the path below instead:
MODEL_PATH = "/content/drive/MyDrive/GROUP_ASSIGNMENT_1/models/q3_models.pkl"


with open(MODEL_PATH, "rb") as f:
    models = cloudpickle.load(f)


vocab = models["vocab"]
unigram_freq = models["unigram_freq"]
bigram_freq = models["bigram_freq"]
unigram_ctx_freq = models["unigram_ctx_freq"]
symspell_index = models["symspell_index"]
K = models["K"]
V = models["V"]

method_a_candidates = models["method_a_candidates"]
method_b_candidates = models["method_b_candidates"]
correct_nonword = models["correct_nonword"]
correct_realword = models["correct_realword"]


# ---------------------------------------------------------
# Correct a complete sentence
# ---------------------------------------------------------

def correct_sentence(sentence, method="B"):

    raw_tokens = sentence.strip().split()

    lower_tokens = [
        ''.join(
            ch
            for ch in token.lower()
            if ch.isalpha()
        )
        for token in raw_tokens
    ]

    corrected = list(raw_tokens)

    changed_flags = [
        False
        for _ in raw_tokens
    ]

    for i, clean_tok in enumerate(lower_tokens):

        if not clean_tok:
            continue

        # -------------------------
        # Non-word error
        # -------------------------

        if clean_tok not in vocab:

            fix = correct_nonword(
                clean_tok,
                method=method
            )

            if fix != clean_tok:

                corrected[i] = fix
                changed_flags[i] = True

        # -------------------------
        # Real-word error
        # -------------------------

        else:

            fix, changed = correct_realword(
                lower_tokens,
                i,
                method=method
            )

            if changed:

                corrected[i] = fix
                changed_flags[i] = True

    return corrected, changed_flags


# ---------------------------------------------------------
# Highlight changed words
# ---------------------------------------------------------

def highlight(
    corrected,
    changed_flags
):

    return ' '.join(
        f"**{word}**"
        if changed
        else word

        for word, changed
        in zip(
            corrected,
            changed_flags
        )
    )


# ---------------------------------------------------------
# Continuous CLI
# ---------------------------------------------------------

def run_cli(method="B"):

    print(
        "=========================================="
    )

    print(
        "Q3 Spelling Corrector CLI"
    )

    print(
        "Type a sentence and press Enter."
    )

    print(
        "Type 'exit' to quit."
    )

    print(
        "=========================================="
    )

    while True:

        text = input(">> ")

        if text.strip().lower() == "exit":

            print("Goodbye!")
            break

        start = time.time()

        corrected, flags = correct_sentence(
            text,
            method=method
        )

        latency_ms = (
            time.time() - start
        ) * 1000

        print(
            "Corrected:",
            highlight(
                corrected,
                flags
            )
        )

        print(
            f"Latency: {latency_ms:.2f} ms"
        )


if __name__ == "__main__":

    run_cli(method="B")

Overwriting q3_cli.py


In [39]:
import cloudpickle
import os

q3_model_bundle = {
    # Corpus/model data
    "vocab": vocab,
    "unigram_freq": unigram_freq,
    "bigram_freq": bigram_freq,
    "unigram_ctx_freq": unigram_ctx_freq,

    # SymSpell index
    "symspell_index": symspell_index,

    # Model settings
    "K": K,
    "V": V,

    # Q3 callable components
    "edits1": edits1,
    "method_a_candidates": method_a_candidates,
    "method_b_candidates": method_b_candidates,
    "correct_nonword": correct_nonword,
    "correct_realword": correct_realword,
}

q3_model_path = os.path.join(
    MODEL_DIR,
    "q3_models.pkl"
)

with open(q3_model_path, "wb") as f:
    cloudpickle.dump(
        q3_model_bundle,
        f
    )

print("Q3 models saved successfully!")
print("Location:", q3_model_path)
print("File exists:", os.path.exists(q3_model_path))

Q3 models saved successfully!
Location: /content/drive/MyDrive/GROUP_ASSIGNMENT_1/models/q3_models.pkl
File exists: True
